# Policy and Trajectory Analysis
Inspect learned policy maps, state visitation, and trajectories.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

import numpy as np
import matplotlib.pyplot as plt

from src.agents.q_learning import QLearningAgent
from src.envs.mountain_car_discrete import make_discrete_env
from src.visualization.policy_maps import plot_policy_heatmap, plot_visitation_heatmap
from src.visualization.trajectories import plot_trajectory_phase_space

In [ ]:
# Load trained Q-learning checkpoint (adjust path if needed).
ckpt = Path('outputs/models/q_learning_discrete_seed42_final.npz')
if not ckpt.exists():
    print('Checkpoint not found:', ckpt)
else:
    agent = QLearningAgent(24, 3, 0.1, 0.99, 1.0, 0.02, 0.995)
    agent.load(ckpt)
    action_map = agent.policy_table
    plot_policy_heatmap(action_map, 'outputs/figures/policy_map_q_learning.png')
    print('Saved policy map to outputs/figures/policy_map_q_learning.png')

In [ ]:
# Collect one greedy trajectory and build a simple visitation map.
env = make_discrete_env(wrappers={'discretize_state': {'n_bins': 24}})
obs, _ = env.reset(seed=42)
positions = []
velocities = []
visits = np.zeros((24, 24), dtype=np.int32)

if 'agent' in globals():
    done = False
    while not done:
        state = tuple(obs)
        visits[state[0], state[1]] += 1
        action = agent.select_action(state, deterministic=True)
        next_obs, reward, terminated, truncated, _ = env.step(action)
        done = bool(terminated or truncated)
        # Approximate back to normalized coordinates for plotting shape only.
        positions.append(state[0])
        velocities.append(state[1])
        obs = next_obs

    plot_visitation_heatmap(visits, 'outputs/figures/visitation_q_learning.png')
    plot_trajectory_phase_space(np.array(positions), np.array(velocities), 'outputs/figures/trajectory_q_learning_phase.png')
    print('Saved visitation and trajectory plots to outputs/figures')
else:
    print('Agent checkpoint is not loaded; train and load first.')

env.close()